In [1]:
import geopandas as gpd
import pandas as pd

vac = pd.read_csv("20_location_vacancy.csv")
raw_geo = pd.read_parquet("shop_period_raw.parquet", columns=["위치ID","경도","위도"]).drop_duplicates("위치ID")
vac_geo = vac.merge(raw_geo, on="위치ID", how="left").dropna(subset=["경도","위도"])

gdf = gpd.GeoDataFrame(vac_geo, geometry=gpd.points_from_xy(vac_geo["경도"], vac_geo["위도"]), crs="EPSG:4326").to_crs(epsg=5181)
poly = gpd.read_file(r"C:\Users\seohg\OneDrive\바탕 화면\2026\seoul datalob contest\area\서울시 상권분석서비스(영역-상권).shp", encoding="utf-8").to_crs(epsg=5181)

joined = gpd.sjoin(gdf, poly[["TRDAR_CD","geometry"]], how="left", predicate="within")
unmatched = joined[joined["TRDAR_CD"].isna()]

# 매칭 실패한 점들이, 가장 가까운 폴리곤에서 몇 미터나 떨어져 있는지 계산
unmatched_pts = gdf[gdf["위치ID"].isin(unmatched["위치ID"])]
nearest = gpd.sjoin_nearest(unmatched_pts, poly[["TRDAR_CD","TRDAR_CD_N","geometry"]], distance_col="거리(m)")

print(nearest["거리(m)"].describe())
print("\n50m 이내:", (nearest["거리(m)"] <= 50).sum())
print("100m 이내:", (nearest["거리(m)"] <= 100).sum())
print("500m 초과:", (nearest["거리(m)"] > 500).sum())

count     88611.000000
mean       8131.396433
std       31876.433840
min           0.260045
25%          22.255941
50%          54.457277
75%         149.591212
max      149474.246691
Name: 거리(m), dtype: float64

50m 이내: 42442
100m 이내: 57488
500m 초과: 8364


In [2]:
import pandas as pd

bins = [0, 50, 100, 200, 500, 1000, 5000, 20000, 200000]
labels = ['0-50m', '50-100m', '100-200m', '200-500m', '500m-1km', '1-5km', '5-20km', '20km+']
nearest['거리구간'] = pd.cut(nearest['거리(m)'], bins=bins, labels=labels, include_lowest=True)

dist = nearest['거리구간'].value_counts().reindex(labels)
print(dist)

# 극단적으로 먼(20km+) 것들이 뭔지 확인 - 이건 좌표 자체가 잘못됐을 가능성
far_outliers = nearest[nearest['거리(m)'] > 20000]
print(f"\n20km 넘게 떨어진 위치 수: {len(far_outliers)}")
print(far_outliers[['위치ID', '거리(m)']].sort_values('거리(m)', ascending=False).head(10))

# 성수동이 어느 구간에 속하는지 직접 확인
raw_all = pd.read_parquet("shop_period_raw.parquet", columns=["위치ID","도로명주소"]).drop_duplicates("위치ID")
seongsu_check = nearest.merge(raw_all, on="위치ID", how="left")
seongsu_check = seongsu_check[seongsu_check["도로명주소"].str.contains("성수", na=False)]
print(f"\n매칭 실패한 성수동 관련 위치 수: {len(seongsu_check)}")
print(seongsu_check[['위치ID','도로명주소','거리(m)','TRDAR_CD_N']].sort_values('거리(m)').head(20))

거리구간
0-50m       42442
50-100m     15046
100-200m    13951
200-500m     8808
500m-1km     2528
1-5km         545
5-20km          0
20km+        5291
Name: count, dtype: int64

20km 넘게 떨어진 위치 수: 5291
                           위치ID          거리(m)
140495  1135010300100260021_5_5  149474.246691
140494  1135010300100260021_5_4  149474.246691
149012  1135010500113130002_2_1  149059.323454
442348  1174011000106700000_2_5  149031.896104
148997  1135010500113130000_2_2  149004.943789
442301  1174011000100180008_1_2  148927.782163
148893  1135010500112670000_3_2  148734.956989
143993  1135010500101400193_1_1  148734.074239
149441  1135010600101370003_2_3  148587.647557
143931  1135010500101360036_1_1  148555.284086

매칭 실패한 성수동 관련 위치 수: 138
                           위치ID                 도로명주소      거리(m) TRDAR_CD_N
7949    1120011500103310157_1_1   서울특별시 성동구 성수이로8길 12   5.307611     경수초등학교
8079    1120011500106970000_1_1   서울특별시 성동구 성수이로2길 10   6.428279  성수119안전센터
8074  1120011500104530002_지하1_1

In [8]:
far = nearest[nearest['거리(m)'] > 20000].copy()

raw_all = pd.read_parquet("shop_period_raw.parquet", columns=["위치ID","경도","위도","도로명주소","시군구명"]).drop_duplicates("위치ID")
far_check = far.merge(raw_all, on="위치ID", how="left")

print(f"20km+ 위치 수: {len(far_check)}")
print("\n경도/위도 분포 (정상 서울 범위: 경도 126.7~127.2, 위도 37.4~37.7):")
print(far_check[["경도_x","위도_x"]].describe())

print("\n표본 10개 - 주소는 서울인데 좌표가 이상한지 확인:")
print(far_check[["위치ID","도로명주소","시군구명","경도_x","위도_x"]].head(10).to_string(index=False))

# 좌표가 특정 값 하나에 몰려있는지(같은 값 반복) 확인
print("\n경도 unique 값 개수:", far_check["경도_x"].nunique())
print("위도 unique 값 개수:", far_check["위도_x"].nunique())

20km+ 위치 수: 5291

경도/위도 분포 (정상 서울 범위: 경도 126.7~127.2, 위도 37.4~37.7):
              경도_x         위도_x
count  5291.000000  5291.000000
mean    128.274725    38.435820
std       0.082248     0.045456
min     128.082318    38.338229
25%     128.205470    38.398229
50%     128.292453    38.433239
75%     128.333124    38.462487
max     128.461839    38.581253

표본 10개 - 주소는 서울인데 좌표가 이상한지 확인:
                     위치ID                  도로명주소 시군구명       경도_x      위도_x
  1111010400100770015_1_1 서울특별시 종로구 자하문로24길 31-6  종로구 128.253518 38.478730
  1111010600100070025_2_1       서울특별시 종로구 효자로 31  종로구 128.255331 38.474722
  1111010600100120000_3_2   서울특별시 종로구 자하문로10길 24  종로구 128.254465 38.475234
  1111010600100250008_1_1 서울특별시 종로구 자하문로6길 11-22  종로구 128.254266 38.474382
  1111010600100350012_2_1     서울특별시 종로구 자하문로6길 6  종로구 128.254092 38.473925
  1111010600100350043_2_1      서울특별시 종로구 자하문로 10  종로구 128.254200 38.472977
  1111010600100470000_1_6     서울특별시 종로구 자하문로2길 3  종로구 128.254419 38.472640
  111101060

In [9]:
raw_all.columns# 이 5,291개가 특정 시군구/법정동에 몰려있는지
print(far_check["시군구명"].value_counts())

# 정상 위치와 비교해서 얼마나 규칙적으로 밀려있는지
# (같은 시군구의 정상 위치 평균 좌표와 비교)
normal_gu_coord = raw_addr_normal = pd.read_parquet(
    "shop_period_raw.parquet", columns=["시군구명","경도","위도"]
)
normal_center = normal_gu_coord[normal_gu_coord["경도"].between(126.7,127.2)].groupby("시군구명")[["경도","위도"]].mean()
print("\n정상 종로구 평균 좌표:")
print(normal_center.loc["종로구"])
print("\n오류난 것들의 평균 좌표:")
print(far_check[["경도_x","위도_x"]].mean())
print("\n오프셋(오류좌표 - 정상좌표):")
print(far_check[["경도_x","위도_x"]].mean().values - normal_center.loc["종로구"].values)
print(far_check.columns)

시군구명
강남구     767
마포구     406
서초구     385
송파구     298
영등포구    276
강서구     268
중구      206
용산구     203
종로구     200
강동구     186
관악구     181
성동구     174
동대문구    172
광진구     164
양천구     155
성북구     146
서대문구    137
노원구     136
중랑구     133
은평구     129
동작구     127
금천구     122
구로구     119
강북구     108
도봉구      93
Name: count, dtype: int64

정상 종로구 평균 좌표:
경도    126.989683
위도     37.576212
Name: 종로구, dtype: float64

오류난 것들의 평균 좌표:
경도_x    128.274725
위도_x     38.435820
dtype: float64

오프셋(오류좌표 - 정상좌표):
[1.28504173 0.85960828]
Index(['위치ID', '첫분기', '끝분기', '관측span분기수', '공실분기수', '공실률', '경도_x', '위도_x',
       'geometry', 'index_right', 'TRDAR_CD', 'TRDAR_CD_N', '거리(m)', '거리구간',
       '경도_y', '위도_y', '도로명주소', '시군구명'],
      dtype='object')


In [10]:
import pandas as pd

# 문제였던 위치ID들
bad_ids = far_check["위치ID"].tolist()

# 이번엔 dedup 안 하고, 그 위치ID의 '모든 분기별' 좌표를 다 가져옴
raw_full = pd.read_parquet(
    "shop_period_raw.parquet", columns=["위치ID", "period", "경도", "위도"]
)
raw_full = raw_full[raw_full["위치ID"].isin(bad_ids)]

# 위치ID별로 좌표가 여러 값(=분기마다 다름)인지 확인
coord_variation = raw_full.groupby("위치ID")[["경도", "위도"]].nunique()
print("좌표값이 2개 이상(분기마다 다른) 위치 수:", (coord_variation["경도"] > 1).sum())
print("좌표값이 1개뿐(항상 똑같이 틀림) 위치 수:", (coord_variation["경도"] == 1).sum())

# 정상 범위(서울) 좌표가 단 하나라도 있는 위치ID 찾기
raw_full["정상범위"] = raw_full["경도"].between(126.7, 127.2) & raw_full["위도"].between(37.4, 37.7)
has_valid = raw_full.groupby("위치ID")["정상범위"].any()

print(f"\n{has_valid.sum()} / {len(has_valid)} 개 위치가 다른 분기에 정상 좌표를 갖고 있음")

좌표값이 2개 이상(분기마다 다른) 위치 수: 4518
좌표값이 1개뿐(항상 똑같이 틀림) 위치 수: 773

4518 / 5291 개 위치가 다른 분기에 정상 좌표를 갖고 있음


In [11]:
import os
print(os.path.exists("22_raw_geo_fixed.csv"))

True


In [12]:
merged = pd.read_csv("21_frequent_with_vacancy_type.csv")
print(merged[merged["TRDAR_CD_N"].str.contains("성수", na=False)])

                          위치ID  관측분기수   평균생존분기수  최장생존분기수  교체횟수       교체율  \
13     1120011500102790042_1_1     18  2.571429        6     6  0.352941   
104    1120011500103150061_4_6     20  3.333333        7     5  0.263158   
136    1120011500103000066_4_1     21  3.500000        5     5  0.250000   
275    1120011500102790050_1_8     28  4.000000       11     6  0.222222   
370    1120011500102790042_2_1     15  3.750000        5     3  0.214286   
...                        ...    ...       ...      ...   ...       ...   
19164  1120011500103150100_1_1     15  7.500000       10     1  0.071429   
20578  1168010700106340003_3_1     15  7.500000       11     1  0.071429   
20579  1168010700106360014_1_2     15  7.500000        8     1  0.071429   
20580  1168010700106360020_3_1     15  7.500000        8     1  0.071429   
20581  1168010700106380009_1_1     15  7.500000        9     1  0.071429   

       자주바뀜여부          경도         위도 시군구명   법정동명                   도로명주소  \
13       Tr

In [13]:
print(merged[merged["TRDAR_CD_N"].str.contains("성수", na=False)][["위치ID","TRDAR_CD_N","시군구명","법정동명"]].drop_duplicates("TRDAR_CD_N"))

                         위치ID  TRDAR_CD_N 시군구명   법정동명
13    1120011500102790042_1_1      성수초등학교  성동구  성수동2가
104   1120011500103150061_4_6         성수역  성동구  성수동2가
372   1120011500108320000_2_1     성수동카페거리  성동구  성수동2가
526   1120011500105640001_1_2   성수119안전센터  성동구  성수동2가
760   1120011500102990182_2_2  성수2가3동주민센터  성동구  성수동2가
1873  1168010700106380007_1_1      성수대교남단  강남구    신사동
7812  1120011400100720008_1_7  성수1가1동주민센터  성동구  성수동1가


In [5]:
import pandas as pd
joined = pd.read_csv("18_frequent_change_with_district.csv")
still_missing = joined[joined["TRDAR_CD"].isna()]
print(len(still_missing))
print(still_missing[["위치ID","경도","위도","시군구명","도로명주소"]])

4305
                            위치ID          경도         위도 시군구명  \
24       1171010900108780000_1_1  127.139923  37.480481  송파구   
30       1171010400101420002_1_3  127.115400  37.508182  송파구   
32       1150010500107430004_1_8  126.823860  37.563974  강서구   
36       1138010100102870015_1_2  126.884601  37.589790  은평구   
41       1129013900101890011_1_5  127.062065  37.611247  성북구   
...                          ...         ...        ...  ...   
20787  1174010900101210083_지하1_1  127.136073  37.539294  강동구   
20789    1174010900102800006_1_3  127.131905  37.543227  강동구   
20794   1174010900104100100_2_12  127.128367  37.539195  강동구   
20798    1174010900104470000_5_3  127.134120  37.535765  강동구   
20801    1174011000100760007_5_3  127.174507  37.572094  강동구   

                       도로명주소  
24       서울특별시 송파구 위례광장로 185  
30         서울특별시 송파구 오금로 194  
32        서울특별시 강서구 마곡서로 133  
36       서울특별시 은평구 수색로 390-4  
41         서울특별시 성북구 한천로 579  
...                      ...  
20787    

In [7]:
import pandas as pd
coords = pd.read_csv("location_coords.csv")  # 19번 실행 후 갱신된 파일

still_failed = coords[coords["경도"].isna()]
print(f"VWorld로도 실패한 위치 수: {len(still_failed):,}")
print(still_failed[["위치ID","시군구명","도로명주소"]])

still_failed.to_csv("vworld_geocoding_failed.csv", index=False, encoding="utf-8-sig")

VWorld로도 실패한 위치 수: 7
                             위치ID  시군구명                 도로명주소
773342    1144010200100790013_4_1   마포구    서울특별시 마포구 만리재옛길 57
773751   1156012100100550018_9_64  영등포구  서울특별시 영등포구 문래로28길 25
773911  1156012100100550018_9_126  영등포구  서울특별시 영등포구 문래로28길 25
774056  1156011000100150028_nan_1  영등포구  서울특별시 영등포구 국회대로66길 9
774459    1144010200100790013_4_2   마포구    서울특별시 마포구 만리재옛길 57
780005  1174010900104520000_지하1_1   강동구   서울특별시 강동구 천호대로 1045
781399  1168010100106470009_15_72   강남구    서울특별시 강남구 테헤란로 131


In [8]:
import geopandas as gpd
import pandas as pd

# 1) 폴리곤 원본을 '변환 전' 그대로 열어서, 도형 자체의 좌표 범위를 확인
poly_raw = gpd.read_file(
    r"C:\Users\seohg\OneDrive\바탕 화면\2026\seoul datalob contest\area\서울시 상권분석서비스(영역-상권).shp",
    encoding="utf-8",
)

print("폴리곤 원본 CRS(좌표계):", poly_raw.crs)
print("\n폴리곤 전체 경계상자(bounding box):")
print(poly_raw.total_bounds)  # [min_x, min_y, max_x, max_y]

# 2) 개별 폴리곤 단위로도 이상한 게 있는지 확인 (하나라도 범위 벗어나면 그 폴리곤이 범인)
bounds = poly_raw.geometry.bounds  # 폴리곤별 minx,miny,maxx,maxy
print("\n폴리곤별 X좌표(경도 or 그 좌표계 X) 범위:")
print(bounds[["minx","maxx"]].describe())
print("\n폴리곤별 Y좌표(위도 or 그 좌표계 Y) 범위:")
print(bounds[["miny","maxy"]].describe())

# 3) 성수 관련 폴리곤만 콕 집어서 확인
seongsu_poly = poly_raw[poly_raw["TRDAR_CD_N"].str.contains("성수", na=False)]
print(f"\n성수 관련 폴리곤 {len(seongsu_poly)}개:")
print(seongsu_poly[["TRDAR_CD_N"]])
print(seongsu_poly.geometry.bounds)

# 4) EPSG:5181로 변환한 후에도 이상한지 재확인
poly_5181 = poly_raw.to_crs(epsg=5181)
print("\n[변환 후] 폴리곤 전체 경계상자(EPSG:5181, 미터 단위):")
print(poly_5181.total_bounds)

폴리곤 원본 CRS(좌표계): EPSG:5181

폴리곤 전체 경계상자(bounding box):
[182079.0203 437149.6674 215472.7203 466040.155 ]

폴리곤별 X좌표(경도 or 그 좌표계 X) 범위:
                minx           maxx
count    1650.000000    1650.000000
mean   198766.106270  199196.887423
std      7280.985653    7281.319375
min    182079.020300  182961.239300
25%    192603.049625  193037.456525
50%    199879.011900  200298.793150
75%    204165.577750  204619.030675
max    215233.365700  215472.720300

폴리곤별 Y좌표(위도 or 그 좌표계 Y) 범위:
                miny           maxy
count    1650.000000    1650.000000
mean   449661.976998  450093.738369
std      5595.452854    5589.349258
min    437149.667400  437659.950100
25%    444962.593475  445386.844800
50%    449618.184100  450023.488200
75%    453331.290825  453775.730175
max    465135.813000  466040.155000

성수 관련 폴리곤 8개:
      TRDAR_CD_N
202   성수역 골목형상점가
211          성수역
217       성수초등학교
228      성수동카페거리
229   성수2가3동주민센터
238    성수119안전센터
252   성수1가1동주민센터
1431      성수대교남단
             minx    

In [9]:
# 성수 관련 미매칭 위치 하나를 골라서, 그 점이 실제로 모든 폴리곤 밖에 있는지 시각적으로 확인
import geopandas as gpd
from shapely.geometry import Point

test_point = Point(127.0550, 37.5440)  # 성수동 대략 좌표, 실제 미매칭 위치로 교체
test_gdf = gpd.GeoDataFrame([1], geometry=[test_point], crs="EPSG:4326").to_crs(epsg=5181)

distances = poly_raw.geometry.distance(test_gdf.geometry.iloc[0])
print("가장 가까운 폴리곤까지 거리:", distances.min(), "미터")
print("그 폴리곤 이름:", poly_raw.loc[distances.idxmin(), "TRDAR_CD_N"])

가장 가까운 폴리곤까지 거리: 0.0 미터
그 폴리곤 이름: 성수역


In [10]:
import pandas as pd

joined = pd.read_csv("20_frequent_change_with_district.csv")
still_missing = joined[joined["TRDAR_CD"].isna()]

print(f"미매칭 위치 수: {len(still_missing)}")
print(still_missing[["위치ID","경도","위도","시군구명","도로명주소"]].head(10))

미매칭 위치 수: 4305
                         위치ID          경도         위도  시군구명  \
24    1171010900108780000_1_1  127.139923  37.480481   송파구   
30    1171010400101420002_1_3  127.115400  37.508182   송파구   
32    1150010500107430004_1_8  126.823860  37.563974   강서구   
36    1138010100102870015_1_2  126.884601  37.589790   은평구   
41    1129013900101890011_1_5  127.062065  37.611247   성북구   
42    1150010500107590005_1_1  126.825844  37.567247   강서구   
50    1171010100101750004_1_2  127.078835  37.511370   송파구   
55  1144011400104380000_지하1_4  126.932223  37.553121   마포구   
56    1156012500100200000_1_3  126.887035  37.522353  영등포구   
65    1156010100106020057_1_1  126.909105  37.513137  영등포구   

                   도로명주소  
24   서울특별시 송파구 위례광장로 185  
30     서울특별시 송파구 오금로 194  
32    서울특별시 강서구 마곡서로 133  
36   서울특별시 은평구 수색로 390-4  
41     서울특별시 성북구 한천로 579  
42   서울특별시 강서구 마곡중앙5로 18  
50     서울특별시 송파구 올림픽로 74  
55    서울특별시 마포구 서강로9길 17  
56  서울특별시 영등포구 영등포로12길 8  
65     서울특별시 영등포구 영신로 16  


In [11]:
from shapely.geometry import Point
import geopandas as gpd

# 위에서 나온 실제 미매칭 위치 하나의 진짜 경도/위도를 여기 넣으세요
real_lon = 127.139923  # still_missing에서 실제 값으로 교체
real_lat = 37.480481

test_point = Point(real_lon, real_lat)
test_gdf = gpd.GeoDataFrame([1], geometry=[test_point], crs="EPSG:4326").to_crs(epsg=5181)

distances = poly_raw.geometry.distance(test_gdf.geometry.iloc[0])
print("가장 가까운 폴리곤까지 거리:", distances.min(), "미터")
print("그 폴리곤 이름:", poly_raw.loc[distances.idxmin(), "TRDAR_CD_N"])

가장 가까운 폴리곤까지 거리: 878.1246604494481 미터
그 폴리곤 이름: 문정동성당


In [ ]:
import pandas as pd

joined = pd.read_csv("20_frequent_change_with_district.csv")

has_trdar = joined[joined["TRDAR_CD_N"].notna()]
no_trdar = joined[joined["TRDAR_CD_N"].isna()]

print(f"TRDAR_CD_N 있음: {len(has_trdar):,}건, 교체율 평균 = {has_trdar['교체율'].mean():.4f}")
print(f"TRDAR_CD_N 없음: {len(no_trdar):,}건, 교체율 평균 = {no_trdar['교체율'].mean():.4f}")

# 두 그룹 요약 통계 비교
summary = joined.assign(상권매칭여부=joined["TRDAR_CD_N"].notna()).groupby("상권매칭여부")["교체율"].describe()
print("\n[그룹별 교체율 요약통계]")
print(summary)

In [14]:
import pandas as pd
vac = pd.read_csv("22_location_vacancy.csv")

# 1) 공실률=0 인 것과 아닌 것 개수
zero_count = (vac["공실률"] == 0).sum()
nonzero = vac[vac["공실률"] > 0]
print(f"공실률=0: {zero_count:,}개")
print(f"공실률>0: {len(nonzero):,}개")

# 2) 0이 아닌 것들만 구간별로 분포 확인
bins = [0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.92]
labels = ['0-5%','5-10%','10-15%','15-20%','20-30%','30-40%','40-50%','50-60%','60-70%','70-80%','80-92%']
nonzero_binned = pd.cut(nonzero["공실률"], bins=bins, labels=labels, include_lowest=True)
dist = nonzero_binned.value_counts().reindex(labels)
print(dist)

공실률=0: 373,726개
공실률>0: 68,802개
공실률
0-5%      16621
5-10%     12514
10-15%     7517
15-20%     6887
20-30%     9080
30-40%     6478
40-50%     4376
50-60%     2794
60-70%     1801
70-80%      631
80-92%      103
Name: count, dtype: int64


In [ ]:
import pandas as pd
full = pd.read_csv("24_full_location_with_district.csv", dtype={"TRDAR_CD": str})
print(full.columns)

Index(['위치ID', '관측분기수', '평균생존분기수', '최장생존분기수', '교체횟수', '교체율', '경도', '위도',
       '시군구명', '도로명주소', '첫분기', '끝분기', '공실률', '대표업종', 'TRDAR_CD', 'TRDAR_CD_N',
       'TRDAR_SE_1', '상권배정여부', '자주바뀜여부'],
      dtype='object')
